# NLTK Stemming Techniques Explained
### Porter, Lancaster, Snowball, Regexp, WordNetLemmatizer | NLP

This notebook demonstrates all five techniques using **NLTK**, with real `nltk.stem` classes as the primary path. If NLTK (or its data packages) is not installed in the environment running this notebook, each section automatically falls back to a small, clearly-labeled from-scratch stand-in so you can still see the *behavior* and compare outputs side by side.

**To get the real NLTK results, run this once:**
```bash
pip install nltk
```
```python
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
```

**Contents**
1. Setup and NLTK availability check
2. Porter Stemmer
3. Lancaster Stemmer
4. Snowball Stemmer (Porter2) + multi-language
5. Regexp Stemmer
6. WordNetLemmatizer (+ POS tagging)
7. Side-by-side comparison table
8. Stemming a full sentence / mini pipeline
9. Vocabulary reduction effect (why it matters for TF-IDF / BoW)
10. Picking the right tool


## 1. Setup and NLTK Availability Check

The cell below tries to import NLTK and download the small data packages the lemmatizer and POS tagger need. `NLTK_OK` tells the rest of the notebook which code path to use.


In [ ]:
import re
import pandas as pd

NLTK_OK = False
try:
    import nltk
    from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
    from nltk.stem.snowball import SnowballStemmer
    from nltk import pos_tag, word_tokenize
    from nltk.corpus import wordnet

    for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

    # Confirm the data actually works, not just that nltk imported
    PorterStemmer().stem('testing')
    WordNetLemmatizer().lemmatize('tests')
    NLTK_OK = True
    print('NLTK is installed and data is available -> using real nltk.stem classes.')
except Exception as e:
    print(f'NLTK not fully available ({type(e).__name__}); using from-scratch fallback implementations instead.')
    print('Run: pip install nltk  and then nltk.download(\'wordnet\'), nltk.download(\'omw-1.4\') for the real thing.')


In [ ]:
# --- Fallback (educational) implementations, used only when NLTK_OK is False ---
# These are simplified stand-ins so the notebook still runs end-to-end everywhere.
# They are NOT the real Porter/Lancaster algorithms -- see nltk_stemming_theory.md for the real rules.

def fallback_porter(word):
    w = word.lower()
    for suf, repl in [('sses', 'ss'), ('ies', 'i'), ('ational', 'ate'), ('tional', 'tion'),
                       ('ization', 'ize'), ('fulness', 'ful'), ('ousness', 'ous'), ('iveness', 'ive'),
                       ('ement', ''), ('able', ''), ('ible', ''), ('eed', 'ee'), ('ing', ''),
                       ('ed', ''), ('ly', ''), ('ies', 'i'), ('es', ''), ('s', '')]:
        if w.endswith(suf) and len(w) - len(suf) >= 2:
            w = w[: -len(suf)] + repl
            # undouble a trailing doubled consonant left behind by stripping -ing/-ed (running -> runn -> run)
            if not repl and len(w) >= 3 and w[-1] == w[-2] and w[-1] not in 'aeiou':
                w = w[:-1]
            return w
    return w

def fallback_lancaster(word):
    # More aggressive: keep stripping known endings repeatedly (iterative, like Lancaster's design)
    w = word.lower()
    endings = ['ational', 'ization', 'fulness', 'ousness', 'iveness', 'ement', 'able', 'ible',
               'ings', 'ing', 'ion', 'ers', 'er', 'est', 'ive', 'ous', 'ful', 'ies', 'ed', 'es', 's', 'y']
    changed = True
    while changed and len(w) > 3:
        changed = False
        for suf in endings:
            if w.endswith(suf) and len(w) - len(suf) >= 3:
                w = w[: -len(suf)]
                changed = True
                break
    return w

def fallback_snowball(word):
    # Treated like a slightly cleaner Porter for this fallback
    return fallback_porter(word)

class FallbackRegexpStemmer:
    def __init__(self, pattern, min_len=2):
        self.pattern = re.compile(pattern)
        self.min_len = min_len

    def stem(self, word):
        m = self.pattern.search(word)
        if m and m.start() >= self.min_len:
            return word[: m.start()]
        return word

_LEMMA_DICT = {
    'studies': 'study', 'studying': 'study', 'better': 'good', 'running': 'run', 'ran': 'run',
    'mice': 'mouse', 'was': 'be', 'were': 'be', 'feet': 'foot', 'cities': 'city', 'geese': 'goose',
    'generously': 'generously', 'university': 'university',
}

def fallback_lemmatize(word, pos='n'):
    if pos == 'v' and word in {'running': 'run', 'ran': 'run', 'was': 'be', 'were': 'be'}:
        return {'running': 'run', 'ran': 'run', 'was': 'be', 'were': 'be'}[word]
    return _LEMMA_DICT.get(word, word)

print('Fallback stemmers ready (used only if NLTK_OK is False).')


## 2. Porter Stemmer

The classic, moderately aggressive, 5-step algorithm. Good general-purpose default for English.


In [ ]:
words = ['running', 'runs', 'ran', 'runner', 'studies', 'studying', 'studied',
         'generously', 'generation', 'relational', 'university', 'flies', 'happily',
         'national', 'connection', 'organization', 'maximum', 'cats', 'ponies']

if NLTK_OK:
    porter = PorterStemmer()
    porter_stems = [porter.stem(w) for w in words]
else:
    porter_stems = [fallback_porter(w) for w in words]

pd.DataFrame({'word': words, 'porter_stem': porter_stems})


## 3. Lancaster Stemmer

Iterative, table-driven, and **much more aggressive** than Porter. Watch how much shorter the stems get.


In [ ]:
if NLTK_OK:
    lancaster = LancasterStemmer()
    lancaster_stems = [lancaster.stem(w) for w in words]
else:
    lancaster_stems = [fallback_lancaster(w) for w in words]

comparison = pd.DataFrame({'word': words, 'porter': porter_stems, 'lancaster': lancaster_stems})
comparison['lancaster_shorter_by'] = comparison['porter'].str.len() - comparison['lancaster'].str.len()
comparison


In [ ]:
print('Average stem length  Porter   :', round(sum(len(s) for s in porter_stems) / len(porter_stems), 2))
print('Average stem length  Lancaster:', round(sum(len(s) for s in lancaster_stems) / len(lancaster_stems), 2))


Lancaster's stems are shorter on average, which can merge more unrelated words together (over-stemming) in exchange for higher recall in search-style tasks.


## 4. Snowball Stemmer (Porter2) + Multi-language Support

Snowball refines Porter's rules and also works for many other languages.


In [ ]:
if NLTK_OK:
    snowball_en = SnowballStemmer('english')
    snowball_stems = [snowball_en.stem(w) for w in words]
    print('Supported Snowball languages:', SnowballStemmer.languages)
else:
    snowball_stems = [fallback_snowball(w) for w in words]
    print('Supported Snowball languages (reference list): english, french, german, spanish, italian, '
          'portuguese, dutch, russian, swedish, finnish, danish, norwegian, ... (NLTK not available to query live)')

pd.DataFrame({'word': words, 'porter': porter_stems, 'snowball': snowball_stems})


In [ ]:
# Snowball vs Porter -- where do they actually differ?
diff_rows = [(w, p, s) for w, p, s in zip(words, porter_stems, snowball_stems) if p != s]
print('Words where Snowball and Porter disagree:')
pd.DataFrame(diff_rows, columns=['word', 'porter', 'snowball'])


In [ ]:
# A quick non-English example (French), when real NLTK is available
if NLTK_OK and 'french' in SnowballStemmer.languages:
    fr = SnowballStemmer('french')
    fr_words = ['manger', 'mangeons', 'mangeait', 'chevaux', 'cheval']
    print(pd.DataFrame({'word': fr_words, 'french_stem': [fr.stem(w) for w in fr_words]}))
else:
    print('Multi-language demo needs real NLTK + Snowball data; skipped here. '
          'With NLTK installed try: SnowballStemmer(\'french\').stem(\'mangeons\')')


## 5. Regexp Stemmer

Fully custom: you supply the regex pattern for what to strip from the end of each word.


In [ ]:
pattern = r'ing$|s$|ed$|ly$|able$|ational$'

if NLTK_OK:
    regexp_stemmer = RegexpStemmer(pattern, min=4)
else:
    regexp_stemmer = FallbackRegexpStemmer(pattern, min_len=4)

regexp_stems = [regexp_stemmer.stem(w) for w in words]
pd.DataFrame({'word': words, 'regexp_stem': regexp_stems})


In [ ]:
# Domain-specific example: strip a custom product-code suffix convention
codes = ['order_v2', 'invoice_v1', 'report_v10', 'summary']
custom_pattern = r'_v\d+$'
if NLTK_OK:
    code_stemmer = RegexpStemmer(custom_pattern, min=3)
else:
    code_stemmer = FallbackRegexpStemmer(custom_pattern, min_len=3)

print([code_stemmer.stem(c) for c in codes])


This is exactly the kind of task a general-purpose stemmer (Porter/Lancaster/Snowball) cannot do out of the box, but `RegexpStemmer` handles in one line.


## 6. WordNetLemmatizer (True Lemmatization)

Unlike the stemmers above, this always returns a **real dictionary word** — but it needs the correct **Part-of-Speech (POS)** to do so accurately.


In [ ]:
lemma_words = [('better', 'a'), ('running', 'v'), ('running', 'n'), ('was', 'v'),
               ('studies', 'v'), ('studies', 'n'), ('geese', 'n'), ('feet', 'n')]

if NLTK_OK:
    lemmatizer = WordNetLemmatizer()
    lemma_results = [(w, pos, lemmatizer.lemmatize(w, pos=pos)) for w, pos in lemma_words]
else:
    lemma_results = [(w, pos, fallback_lemmatize(w, pos)) for w, pos in lemma_words]

pd.DataFrame(lemma_results, columns=['word', 'pos_given', 'lemma'])


Notice: `('running', 'n')` stays `"running"` (treated as a noun, e.g. "the running of the bulls"), while `('running', 'v')` correctly becomes `"run"`. This is why POS matters.


In [ ]:
# Automatic POS tagging -> feed the right tag into the lemmatizer
def penn_to_wordnet(tag):
    if tag.startswith('J'):
        return 'a'
    if tag.startswith('V'):
        return 'v'
    if tag.startswith('R'):
        return 'r'
    return 'n'  # default: noun

sentence = "The striped bats were hanging on their feet and eating best fruits"

if NLTK_OK:
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(w, pos=penn_to_wordnet(t)) for w, t in tagged]
else:
    tokens = sentence.split()
    # a tiny manual tag set just for this sentence, since a real POS tagger isn't available
    manual_tags = {'striped': 'J', 'bats': 'N', 'were': 'V', 'hanging': 'V', 'eating': 'V', 'best': 'J', 'fruits': 'N'}
    tagged = [(w, manual_tags.get(w, 'N')) for w in tokens]
    lemmas = [fallback_lemmatize(w, pos=penn_to_wordnet(t)) for w, t in tagged]

pd.DataFrame({'token': tokens, 'pos_tag': [t for _, t in tagged], 'lemma': lemmas})


## 7. Side-by-Side Comparison: All Techniques on the Same Words


In [ ]:
if NLTK_OK:
    wn_lemmas_verb = [lemmatizer.lemmatize(w, pos='v') for w in words]
else:
    wn_lemmas_verb = [fallback_lemmatize(w, pos='v') for w in words]

full_comparison = pd.DataFrame({
    'word': words,
    'porter': porter_stems,
    'lancaster': lancaster_stems,
    'snowball': snowball_stems,
    'regexp': regexp_stems,
    'wordnet_lemma(verb)': wn_lemmas_verb,
})
full_comparison


In [ ]:
import matplotlib.pyplot as plt

avg_len = {
    'Porter': sum(len(s) for s in porter_stems) / len(porter_stems),
    'Lancaster': sum(len(s) for s in lancaster_stems) / len(lancaster_stems),
    'Snowball': sum(len(s) for s in snowball_stems) / len(snowball_stems),
    'Regexp': sum(len(s) for s in regexp_stems) / len(regexp_stems),
    'WordNet lemma': sum(len(s) for s in wn_lemmas_verb) / len(wn_lemmas_verb),
}

plt.figure(figsize=(8, 4.5))
plt.bar(avg_len.keys(), avg_len.values(), color='steelblue')
plt.ylabel('Average output length (characters)')
plt.title('Aggressiveness Comparison: Shorter Output = More Aggressive')
plt.xticks(rotation=15)
plt.show()


## 8. Stemming a Full Sentence (Mini Pipeline)

A typical preprocessing step: tokenize, then stem every token.


In [ ]:
def stem_sentence(sentence, stemmer_fn):
    tokens = re.findall(r"[a-zA-Z']+", sentence.lower())
    return [stemmer_fn(t) for t in tokens]

sample = "The researchers were studying how organizations communicate effectively during national emergencies"

if NLTK_OK:
    porter_fn, lancaster_fn, snowball_fn = porter.stem, lancaster.stem, snowball_en.stem
else:
    porter_fn, lancaster_fn, snowball_fn = fallback_porter, fallback_lancaster, fallback_snowball

print('Original :', sample)
print('Porter   :', ' '.join(stem_sentence(sample, porter_fn)))
print('Lancaster:', ' '.join(stem_sentence(sample, lancaster_fn)))
print('Snowball :', ' '.join(stem_sentence(sample, snowball_fn)))


## 9. Why This Matters: Vocabulary Reduction for BoW / TF-IDF

Stemming shrinks the vocabulary used by classical models (Bag of Words, TF-IDF), which reduces sparsity and lets the model recognize that "run", "running" and "runs" are related.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "I love running every morning",
    "She runs five miles daily",
    "He ran a marathon last year",
    "They are running a new business",
]

if NLTK_OK:
    stem_docs = [' '.join(porter.stem(w) for w in re.findall(r'\w+', d.lower())) for d in docs]
else:
    stem_docs = [' '.join(fallback_porter(w) for w in re.findall(r'\w+', d.lower())) for d in docs]

cv_raw = CountVectorizer()
cv_raw.fit(docs)
cv_stem = CountVectorizer()
cv_stem.fit(stem_docs)

print('Vocabulary size WITHOUT stemming:', len(cv_raw.get_feature_names_out()))
print('Vocabulary size WITH Porter stemming:', len(cv_stem.get_feature_names_out()))
print()
print('Stemmed documents:')
for d in stem_docs:
    print(' ', d)


Notice `run`, `runs`, `running`, `ran` (word forms of "run") collapse toward a shared stem, which is exactly why stemming before building a BoW/TF-IDF matrix reduces the feature space and helps a classical model generalize across these forms.


## 10. Picking the Right Tool — Quick Reference

| Need | Use |
|---|---|
| Fast, general-purpose English baseline | `PorterStemmer` or `SnowballStemmer('english')` |
| Maximum aggressiveness / recall, don't mind noisier stems | `LancasterStemmer` |
| Non-English text | `SnowballStemmer(<language>)` |
| Custom domain suffix rules (IDs, jargon) | `RegexpStemmer(pattern, min=...)` |
| Output must be a real, readable word | `WordNetLemmatizer` (with a POS tagger feeding it) |

See `nltk_stemming_theory.md` for the full explanation of each algorithm's internal rules.
